# Collecting Multi-Daerah OCDS Data
Notebook ini mengumpulkan data raw OCDS untuk semua daerah target dan membangun satu master CSV mentah.


## 1. Define Library and Configuration

In [1]:
import csv
import json
import time
import zipfile
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

base_url = 'https://opentender.net/api/tender/export-ocds-batch/'
years = list(range(2015, 2025))
target_lspe = {
    '127': 'Jakarta',
    '15': 'Jawa Timur',
    '42': 'Jawa Tengah'
}

request_timeout = 120
sleep_seconds = 1

json_root = Path('../data/json')
csv_root = Path('../data/csv')
master_csv_path = csv_root / 'master_data_ocds.csv'
summary_csv_path = csv_root / 'raw_data_summary_by_daerah.csv'

fieldnames = [
    'lspe_id', 'nama_daerah', 'source_year', 'source_file', 'ocid', 'release_id', 'date',
    'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainProcurementCategory',
    'tender_minValue', 'tender_datePublished', 'tender_status', 'award_id', 'award_title',
    'award_date', 'award_value', 'award_supplier'
]

json_root.mkdir(parents=True, exist_ok=True)
csv_root.mkdir(parents=True, exist_ok=True)

print('Target daerah:')
display(pd.DataFrame([{'lspe_id': key, 'nama_daerah': value} for key, value in target_lspe.items()]))
print(f'Years covered: {years[0]} - {years[-1]}')
print('Notebook mode: always redownload and rewrite raw files')


Target daerah:


,lspe_id,nama_daerah
0,127,Jakarta
1,15,Jawa Timur
2,42,Jawa Tengah


Years covered: 2015 - 2024
Notebook mode: always redownload and rewrite raw files


## 2. Download Helper

In [2]:
def expected_json_path(lspe_id, year):
    return json_root / lspe_id / f'ocds-data-tender-{year}-{lspe_id}.json'

def download_json_batch(lspe_id, nama_daerah, year):
    target_path = expected_json_path(lspe_id, year)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    zip_path = Path(f'ocds_{lspe_id}_{year}.zip')
    target_url = f'{base_url}?year={year}&lpse={lspe_id}'
    print(f'[download] {nama_daerah} {year}')

    response = requests.get(target_url, stream=True, timeout=request_timeout)
    response.raise_for_status()

    with open(zip_path, 'wb') as file_handle:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file_handle.write(chunk)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        json_members = [member for member in zip_ref.namelist() if member.endswith('.json')]
        if not json_members:
            raise ValueError(f'No JSON file found inside {zip_path.name}')

        member = json_members[0]
        zip_ref.extract(member, target_path.parent)
        extracted_path = target_path.parent / Path(member).name

        if target_path.exists():
            target_path.unlink()

        extracted_path.rename(target_path)

    zip_path.unlink(missing_ok=True)
    time.sleep(sleep_seconds)
    return target_path


## 3. Download All Files

In [3]:
for lspe_id, nama_daerah in target_lspe.items():
    print('=' * 70)
    print(f'DOWNLOADING {nama_daerah} ({lspe_id})')
    print('=' * 70)
    for year in years:
        try:
            download_json_batch(lspe_id, nama_daerah, year)
        except Exception as exc:
            print(f'[error] {nama_daerah} {year}: {exc}')


DOWNLOADING Jakarta (127)
[download] Jakarta 2015
[download] Jakarta 2016
[download] Jakarta 2017
[download] Jakarta 2018
[download] Jakarta 2019
[download] Jakarta 2020
[download] Jakarta 2021
[download] Jakarta 2022
[download] Jakarta 2023
[download] Jakarta 2024
[error] Jakarta 2024: 404 Client Error: Not Found for url: https://opentender.net/api/tender/export-ocds-batch/?year=2024&lpse=127
DOWNLOADING Jawa Timur (15)
[download] Jawa Timur 2015
[download] Jawa Timur 2016
[download] Jawa Timur 2017
[download] Jawa Timur 2018
[download] Jawa Timur 2019
[download] Jawa Timur 2020
[download] Jawa Timur 2021
[download] Jawa Timur 2022
[download] Jawa Timur 2023
[download] Jawa Timur 2024
DOWNLOADING Jawa Tengah (42)
[download] Jawa Tengah 2015
[download] Jawa Tengah 2016
[download] Jawa Tengah 2017
[download] Jawa Tengah 2018
[download] Jawa Tengah 2019
[download] Jawa Tengah 2020
[download] Jawa Tengah 2021
[download] Jawa Tengah 2022
[download] Jawa Tengah 2023
[download] Jawa Tengah 2

## 4. Convert JSON into One Raw Master CSV

In [4]:
total_rows_global = 0
summary_rows = []

with open(master_csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for lspe_id, nama_daerah in target_lspe.items():
        region_dir = json_root / lspe_id
        files = sorted(region_dir.glob(f'ocds-data-tender-*-{lspe_id}.json'))

        if not files:
            print(f'[warning] no files found for {nama_daerah} ({lspe_id})')
            summary_rows.append({'lspe_id': lspe_id, 'nama_daerah': nama_daerah, 'json_files': 0, 'rows': 0, 'min_award_date': None, 'max_award_date': None})
            continue

        print('=' * 70)
        print(f'PROCESSING {nama_daerah} ({lspe_id})')
        print('=' * 70)

        rows_per_region = 0
        award_dates = []

        for file_path in files:
            try:
                payload = json.loads(file_path.read_text(encoding='utf-8'))
            except Exception as exc:
                print(f'[warning] failed reading {file_path.name}: {exc}')
                continue

            source_year = file_path.stem.split('-')[-2]
            for release in payload.get('releases', []):
                tender = release.get('tender', {}) or {}
                buyer = release.get('buyer', {}) or {}
                awards = release.get('awards', []) or []

                for award in awards:
                    suppliers = award.get('suppliers', []) or []
                    supplier_names = ', '.join([supplier.get('name', '') for supplier in suppliers if supplier.get('name')]) or None

                    writer.writerow({
                        'lspe_id': lspe_id,
                        'nama_daerah': nama_daerah,
                        'source_year': source_year,
                        'source_file': file_path.name,
                        'ocid': release.get('ocid'),
                        'release_id': release.get('id'),
                        'date': release.get('date'),
                        'buyer_name': buyer.get('name'),
                        'buyer_id': buyer.get('id'),
                        'tender_id': tender.get('id'),
                        'tender_title': tender.get('title'),
                        'mainProcurementCategory': tender.get('mainProcurementCategory'),
                        'tender_minValue': (tender.get('minValue') or {}).get('amount'),
                        'tender_datePublished': tender.get('datePublished'),
                        'tender_status': tender.get('status'),
                        'award_id': award.get('id'),
                        'award_title': award.get('title'),
                        'award_date': award.get('date'),
                        'award_value': (award.get('value') or {}).get('amount'),
                        'award_supplier': supplier_names
                    })

                    if award.get('date'):
                        award_dates.append(award.get('date'))

                    rows_per_region += 1
                    total_rows_global += 1

        print(f'Rows written: {rows_per_region:,}')
        print(f'Files found : {len(files)}')

        summary_rows.append({
            'lspe_id': lspe_id,
            'nama_daerah': nama_daerah,
            'json_files': len(files),
            'rows': rows_per_region,
            'min_award_date': min(award_dates) if award_dates else None,
            'max_award_date': max(award_dates) if award_dates else None
        })


PROCESSING Jakarta (127)
Rows written: 8,678
Files found : 9
PROCESSING Jawa Timur (15)
Rows written: 6,440
Files found : 10
PROCESSING Jawa Tengah (42)
Rows written: 6,038
Files found : 9


## 5. Final Summary and Save Diagnostics

In [5]:
summary_df = pd.DataFrame(summary_rows).sort_values(['nama_daerah', 'lspe_id']).reset_index(drop=True)
summary_df.to_csv(summary_csv_path, index=False)

print('RAW COLLECTION SUMMARY')
print('=' * 70)
display(summary_df)

print('Saved files:')
print(f'- Master CSV  : {master_csv_path.resolve()}')
print(f'- Summary CSV : {summary_csv_path.resolve()}')
print(f'- Total rows  : {total_rows_global:,}')


RAW COLLECTION SUMMARY


,lspe_id,nama_daerah,json_files,rows,min_award_date,max_award_date
0,127,Jakarta,9,8678,2014-12-24T00:00:00.000000Z,2023-10-18T00:00:00.000000Z
1,42,Jawa Tengah,9,6038,2014-11-29T00:00:00.000000Z,2023-10-16T00:00:00.000000Z
2,15,Jawa Timur,10,6440,2014-12-19T00:00:00.000000Z,2024-10-02T15:00:00.000000Z


Saved files:
- Master CSV  : C:\Users\VICTUS\coding\collab\03_training\data\csv\master_data_ocds.csv
- Summary CSV : C:\Users\VICTUS\coding\collab\03_training\data\csv\raw_data_summary_by_daerah.csv
- Total rows  : 21,156
